In [ ]:
import pandas as pd
import os

In [ ]:
ratings = pd.read_csv('../data/preprocessed/ratings_cleaned.csv')
movies = pd.read_csv('../data/preprocessed/movies_cleaned.csv')

In [ ]:
ratings.head()

In [ ]:
user_item_matrix = ratings.pivot(
    index='movieId',
    columns='userId',
    values='rating'
).fillna(0)

In [ ]:
user_item_matrix

In [ ]:
user_item_matrix.shape

In [ ]:
output_dir = '../data/preprocessed'

In [ ]:
os.makedirs(output_dir, exist_ok=True)

In [ ]:
output_path = f'{output_dir}/user_item_matrix_baseline.pkl'
user_item_matrix.to_pickle(output_path)

In [ ]:
output_path

## LightFM

In [ ]:
from lightfm.data import Dataset
import scipy.sparse as sp
import pickle

In [ ]:
dataset = Dataset()

In [ ]:
all_genres = set(g for sublist in movies['genres'] for g in sublist)

In [ ]:
dataset.fit(
    users=ratings['userId'].unique(),
    items=movies['movieId'].unique(),
    item_features=all_genres
)

In [ ]:
print(f"Dataset dimensions: {dataset.interactions_shape()}")

In [ ]:
(interactions, weights) = dataset.build_interactions(
    ((row['userId'], row['movieId'], row['rating']) for index, row in ratings.iterrows())
)

In [ ]:
item_features = dataset.build_item_features(
    ((row['movieId'], row['genres']) for index, row in movies.iterrows())
)

In [ ]:
with open('../data/preprocessed/lightfm_dataset.pkl', 'wb') as f:
    pickle.dump(dataset, f)
    
sp.save_npz('../data/preprocessed/lightfm_interactions.npz', interactions)
sp.save_npz('../data/preprocessed/lightfm_weights.npz', weights)
sp.save_npz('../data/preprocessed/lightfm_item_features.npz', item_features)